# Cenário aplicado 8D — NSGA-III e MOEA/D corrigidos sob orçamento igual

Calibra e executa os dois EAs com o orçamento integral do CNBI (19.830 avaliações por execução), reparo esférico comum e direções Das–Dennis candidatas p=2,3,4. A calibração usa sementes 1010, 2110 e 3070; as fronteiras finais usam sementes pareadas 1–10. Todos os artefatos são identificados por fingerprint e retomáveis por arquivo de execução.

In [ ]:
from pathlib import Path
import hashlib,json,time
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import qmc
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from pymoo.core.problem import Problem
from pymoo.core.repair import Repair
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.optimize import minimize as pmin
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
def root(start=Path.cwd()):
    for p in (start.resolve(),*start.resolve().parents):
        if (p/'AGENTS.md').exists(): return p
    raise FileNotFoundError('Raiz não encontrada')
ROOT=root(); HOME=Path.home(); OUT=ROOT/'results'/'applied'/'applied_8d_equal_budget'; RUNS=OUT/'runs'; TUNE=OUT/'tuning'; FINAL=OUT/'final_fronts'
for p in (OUT,RUNS,TUNE,FINAL): p.mkdir(parents=True,exist_ok=True)
EXCEL=HOME/'Documents'/'Dissertação'/'02_REFERENCIAS'/'Dados'/'VRF_artigo.xlsx'; PSTAR=ROOT/'data'/'reference_fronts'/'referencia_pareto_8D_nsga3_moead.csv'
XCOLS=['cs','f','md']; YCOLS=['T','MTTF','WR','Ra','Rt','Kp','ROI','OEE']; SIGNS=np.array([-1,-1,1,1,1,1,-1,-1.]); ALPHA=2**.75; BUDGET=19830; CAL_SEEDS=(1010,2110,3070); FINAL_SEEDS=tuple(range(1,11))
d=pd.read_excel(EXCEL); d.columns=[str(c).strip() for c in d.columns]; model=Pipeline([('poly',PolynomialFeatures(2,include_bias=False)),('reg',LinearRegression())]).fit(d[XCOLS].to_numpy(float),d[YCOLS].to_numpy(float))
P=pd.read_csv(PSTAR)[YCOLS].to_numpy(float)*SIGNS; ideal=P.min(0); amp=np.maximum(P.max(0)-ideal,1e-12); Pn=(P-ideal)/amp; treeP=cKDTree(Pn)
cfg_base={'budget':BUDGET,'alpha':ALPHA,'calibration_seeds':CAL_SEEDS,'final_seeds':FINAL_SEEDS,'excel_sha256':hashlib.sha256(EXCEL.read_bytes()).hexdigest(),'pstar_sha256':hashlib.sha256(PSTAR.read_bytes()).hexdigest(),'schema':1}


In [ ]:
def project(X):
    X=np.asarray(X,float); n=np.linalg.norm(X,axis=1,keepdims=True); return X*np.minimum(1,ALPHA/np.maximum(n,1e-15))
class SphereRepair(Repair):
    def _do(self,problem,X,**kwargs): return project(X)
class AppliedProblem(Problem):
    def __init__(self): super().__init__(n_var=3,n_obj=8,xl=-ALPHA,xu=ALPHA); self.actual_evaluations=0
    def _evaluate(self,X,out,*args,**kwargs):
        violation=np.max(np.linalg.norm(X,axis=1)-ALPHA)
        if violation>1e-8: raise RuntimeError(f'Reparo esférico falhou: {violation}')
        self.actual_evaluations+=len(X); out['F']=model.predict(X)*SIGNS
def fingerprint(body): return hashlib.sha256(json.dumps(body,sort_keys=True,default=list).encode()).hexdigest()
def params_id(p): return '_'.join(f'{k}-{str(v).replace(".","p")}' for k,v in sorted(p.items()))
def make_alg(method,params,dirs):
    repair=SphereRepair(); cross=SBX(prob=float(params['sbx_probability']),eta=float(params['sbx_eta']),repair=repair); mut=PM(prob=1/3,eta=float(params['pm_eta']),repair=repair)
    if method=='NSGAIII': return NSGA3(ref_dirs=dirs,pop_size=len(dirs),crossover=cross,mutation=mut,repair=repair,eliminate_duplicates=True)
    nn=max(2,min(len(dirs)-1,round(float(params['neighbor_fraction'])*len(dirs)))); return MOEAD(ref_dirs=dirs,n_neighbors=nn,prob_neighbor_mating=float(params['prob_neighbor_mating']),crossover=cross,mutation=mut,repair=repair)
def sanitize(X):
    X=project(np.asarray(X,float)); _,i=np.unique(np.round(X,10),axis=0,return_index=True); X=X[np.sort(i)]; F=model.predict(X); idx=NonDominatedSorting(method='efficient_non_dominated_sort').do(F*SIGNS,only_non_dominated_front=True); return X[idx],F[idx]
def igd(F): return float(cKDTree((F*SIGNS-ideal)/amp).query(Pn,k=1,workers=-1)[0].mean())
U=qmc.Sobol(8,scramble=True,seed=8899).random_base2(12); HV_REF=np.full(8,1.1); Z=U*HV_REF
def hv(F):
    A=(F*SIGNS-ideal)/amp; hit=np.zeros(len(Z),bool)
    for s in range(0,len(Z),512): hit[s:s+512]=np.any(np.all(A[:,None,:]<=Z[None,s:s+512,:],2),0)
    return float(hit.mean()*np.prod(HV_REF))
def run(method,params,seed,stage):
    body={**cfg_base,'method':method,'params':params,'seed':seed,'stage':stage}; fid=fingerprint(body); path=RUNS/f'{stage}_{method}_{params_id(params)}_seed{seed}_{fid[:12]}.npz'
    if path.exists():
        z=np.load(path,allow_pickle=False); return np.asarray(z['X']),np.asarray(z['F']),json.loads(str(z['meta']))
    dirs=get_reference_directions('das-dennis',8,n_partitions=int(params['n_partitions'])); pop=len(dirs); ngen=max(2,BUDGET//pop); problem=AppliedProblem(); alg=make_alg(method,params,dirs); t=time.perf_counter(); res=pmin(problem,alg,('n_gen',ngen),seed=int(seed),verbose=False,save_history=False); elapsed=time.perf_counter()-t
    Xraw=np.asarray(res.pop.get('X'),float); X,F=sanitize(Xraw); meta={'identity':body,'fingerprint':fid,'pop_size':pop,'generations':ngen,'evaluations':problem.actual_evaluations,'budget':BUDGET,'wall_seconds':elapsed,'n_final':len(X)}
    if problem.actual_evaluations>BUDGET: raise RuntimeError(f'Orçamento excedido: {problem.actual_evaluations}')
    np.savez_compressed(path,X=X,F=F,meta=json.dumps(meta)); return X,F,meta
def evaluate(method,candidates,seeds,stage):
    rows=[]
    for p in candidates:
        for s in seeds:
            X,F,m=run(method,p,s,stage); rows.append({'method':method,'stage':stage,'config':params_id(p),'seed':s,**p,**{k:m[k] for k in ('pop_size','generations','evaluations','wall_seconds','n_final')},'IGD1':igd(F),'HV':hv(F)})
    return rows
def rank(rows): return pd.DataFrame(rows).groupby('config',as_index=False).agg(IGD1_median=('IGD1','median'),IGD1_IQR=('IGD1',lambda x:x.quantile(.75)-x.quantile(.25)),HV_median=('HV','median'),wall_median=('wall_seconds','median')).sort_values(['IGD1_median','HV_median','IGD1_IQR','wall_median'],ascending=[True,False,True,True])


In [ ]:
base={'sbx_probability':1.,'sbx_eta':20.,'pm_eta':20.,'neighbor_fraction':.2,'prob_neighbor_mating':.9}; allrows=[]; selected={}
for method in ('NSGAIII','MOEAD'):
    A=[{'n_partitions':p,**base} for p in (2,3,4)]; ra=evaluate(method,A,CAL_SEEDS,'A'); allrows+=ra; top=rank(ra).head(2).config.tolist(); pbest=[int(next(x['n_partitions'] for x in ra if x['config']==c)) for c in top]
    specs=[(pbest[i%2],(.9,1.)[(i//2)%2],(10.,20.,30.)[i%3],(15.,20.,30.)[(i//3+i)%3]) for i in range(12)]; B=[{'n_partitions':p,'sbx_probability':sp,'sbx_eta':se,'pm_eta':pe,'neighbor_fraction':.2,'prob_neighbor_mating':.9} for p,sp,se,pe in specs]; rb=evaluate(method,B,CAL_SEEDS,'B'); allrows+=rb; win=rank(rb).iloc[0].config; proto=next(x for x in rb if x['config']==win); best={k:proto[k] for k in ('n_partitions','sbx_probability','sbx_eta','pm_eta','neighbor_fraction','prob_neighbor_mating')}
    if method=='MOEAD':
        CANDS=[{**best,'neighbor_fraction':nf,'prob_neighbor_mating':pm} for nf in (.1,.2,.3) for pm in (.7,.9,1.)]; rc=evaluate(method,CANDS,CAL_SEEDS,'C'); allrows+=rc; win=rank(rc).iloc[0].config; proto=next(x for x in rc if x['config']==win); best={k:proto[k] for k in best}
    selected[method]=best
pd.DataFrame(allrows).to_csv(TUNE/'calibration_results.csv',index=False); (TUNE/'selected_parameters.json').write_text(json.dumps(selected,indent=2),encoding='utf-8'); print(json.dumps(selected,indent=2))


In [ ]:
manifest=[]
for method in ('NSGAIII','MOEAD'):
    for seed in FINAL_SEEDS:
        X,F,m=run(method,selected[method],seed,'FINAL'); name=('nsga3' if method=='NSGAIII' else 'moead')+f'_seed{seed:03d}.csv'; out=pd.DataFrame(X,columns=XCOLS); out[YCOLS]=F; out.to_csv(FINAL/name,index=False); manifest.append({'method':method,'seed':seed,'file':name,**m})
man=pd.DataFrame(manifest); man.to_csv(OUT/'final_manifest.csv',index=False); (OUT/'run_manifest.json').write_text(json.dumps({**cfg_base,'selected':selected,'final_directory':str(FINAL)},indent=2,default=list),encoding='utf-8'); display(man); print('Fronteiras finais:',FINAL)
